In [3]:
import boto3
import requests
import csv
import io
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor

BUCKET = "cultivate-mapping-data"
INPUT_KEY = "raw/exploration_data/2026_data/04_SHARECITY100/dead_link_report.csv"
OUTPUT_KEY = "raw/exploration_data/2026_data/04_SHARECITY100/scraped_text.csv"

s3 = boto3.client("s3", region_name="eu-north-1")

In [4]:
body = s3.get_object(Bucket=BUCKET, Key=INPUT_KEY)["Body"].read().decode("utf-8")
reader = csv.DictReader(io.StringIO(body))
alive_urls = [(row["city"], row["name"], row["url"]) for row in reader if row["alive"] == "True"]
print(f"Alive URLs to scrape: {len(alive_urls)}")

Alive URLs to scrape: 5292


In [5]:
def scrape_text(url, timeout=15):
    try:
        r = requests.get(url, timeout=timeout, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        text = soup.get_text(separator=" ", strip=True)
        return text[:5000]
    except Exception:
        return ""

# test with first 3
for city, name, url in alive_urls[:3]:
    text = scrape_text(url)
    print(f"{city} | {name} | {len(text)} chars")
    print(text[:200])
    print("---")

Nairobi | Kahurura Community Garden | 0 chars

---
Nairobi | Frī mā mīlā | 25 chars
Free My Meal KE | Nairobi
---
Nairobi | Soko la Wakulima wa Nairobi | 3273 chars
Nairobi Farmers Market: A Fresh Start for Our Community – World Farmers Markets Coalition Skip to Content Home MAMi Nairobi Farmers Market: A Fresh Start for Our Community Share 1 The countdown has be
---


In [6]:
# scrape all alive URLs (parallel, 10 workers)
with ThreadPoolExecutor(max_workers=10) as pool:
    texts = list(pool.map(lambda x: scrape_text(x[2]), alive_urls))

print(f"Done: {len(texts)} URLs scraped")
print(f"Empty: {sum(1 for t in texts if not t)}/{len(texts)}")

Done: 5292 URLs scraped
Empty: 555/5292


In [7]:
# save to S3
output = io.StringIO()
writer = csv.writer(output)
writer.writerow(["city", "name", "url", "scraped_text"])

for (city, name, url), text in zip(alive_urls, texts):
    writer.writerow([city, name, url, text])

s3.put_object(Bucket=BUCKET, Key=OUTPUT_KEY, Body=output.getvalue().encode("utf-8"))
print(f"Saved to s3://{BUCKET}/{OUTPUT_KEY}")

Saved to s3://cultivate-mapping-data/raw/exploration_data/2026_data/04_SHARECITY100/scraped_text.csv
